# PathSnap P1 — dataset and feature review

Run `python -m pathsnap_p1 prepare` first. This notebook reads actual generated results; it does not fit preprocessing or access evaluator data. Read STEP_BY_STEP_GUIDE.md and HANDOVER.md alongside it.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, IFrame
root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
out = root / 'outputs'
inventory = pd.read_csv(out / 'dataset_inventory.csv', dtype={'trip_id': str})
sweep = pd.read_csv(out / 'candidate_sweep_summary.csv')
display(inventory.groupby('split').agg(trips=('trip_id','size'), gps=('raw_observations','sum'), candidate_rows=('candidate_rows','sum')))


## Raw file meanings and cleaning

`.track`: longitude, latitude, timestamp; `.nodes`: longitude, latitude; `.arcs`: node connections; `.route`: ordered reference arc IDs. Original row IDs are retained. Exact duplicate GPS records are removed; repeated locations at different times remain.

In [ ]:
display(inventory)
display(inventory[['split','trip_id','invalid_observations','exact_duplicates','duplicate_timestamps','large_gaps','route_breaks']].sort_values('large_gaps', ascending=False).head(15))
display(Image(filename=str(out / 'figures/dataset_summary.png')))
display(Image(filename=str(out / 'figures/coordinates_and_intervals.png')))


## Candidate coverage

All distances are metres in a suitable per-trip UTM CRS. Radius sets may be empty. Top-k is exact finite-segment distance. Reference candidates are never inserted. Scores use derived monotonic-alignment labels.

In [ ]:
display(sweep)
display(Image(filename=str(out / 'figures/candidate_miss_rates.png')))


## Trace one observation

Rows share a GPS observation and differ by candidate road. Keep target/quality metadata out of the model feature matrix.

In [ ]:
first = inventory.loc[inventory.split.eq('train')].iloc[0]
trip_dir = out / 'processed' / 'train' / first.trip_id
candidates = pd.read_parquet(trip_dir / 'candidates.parquet')
observations = pd.read_parquet(trip_dir / 'observations.parquet')
display(observations.head())
display(candidates.loc[candidates.observation_id.eq(observations.observation_id.iloc[0])])
display(json.loads((out / 'feature_schema.json').read_text()))


## Geographical review

The interactive maps require Folium at generation time and internet for external browser assets. Static figures use real projected geometry. Reference layers are only for development data.

In [ ]:
display(Image(filename=str(out / 'figures/train_00000001_trajectory.png')))
map_path = out / 'maps/train_00000001.html'
if map_path.exists():
    display(IFrame(src='../outputs/maps/train_00000001.html', width='100%', height=550))
else:
    print('Run python -m pathsnap_p1 maps after installing Folium.')


## Actual model comparisons

Run `python -m pathsnap_p1 train` first. Candidate macro F1 is binary; observation Top-1 includes candidate misses. Training imputation/scaling are fitted inside each trip-grouped CV fold.

In [ ]:
comparison = out / 'model_comparison.csv'
if comparison.exists():
    display(pd.read_csv(comparison))
    display(pd.read_csv(out / 'cv_results.csv'))
    display(Image(filename=str(out / 'figures/model_comparison.png')))
else:
    print('Model experiments have not completed yet.')
